In [18]:
import re
import string
import nltk
from nltk.corpus import stopwords
nltk.download('punkt')
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
from nltk import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
nltk.download('wordnet')
nltk.download('omw-1.4')
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [19]:
data= pd.read_csv('sentences.csv')
data.head()

,id,sentiment,sentence
0,1,1,The weather today is absolutely beautiful and ...
1,2,0,I can't believe how terrible the service was a...
2,3,1,She passed her exam with flying colors and fel...
3,4,0,The project failed miserably due to poor plann...
4,5,1,Every morning I wake up feeling grateful and f...


In [20]:
def preprocessing_pipeline(text):
    text=text.lower()
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F" 
        "\U0001F300-\U0001F5FF" 
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001f926-\U0001f937"
        "\U00010000-\U0010ffff"
        "\u2640-\u2642"
        "\u2600-\u2B55"
        "\u200d"
        "\u23cf"
        "\u23e9"
        "\u231a"
        "\ufe0f" 
        "\u3030"
        "]+", re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    preprocessed_text = ' '.join(lemmatized_tokens)
    return preprocessed_text
data['preprocessed_text'] = data['sentence'].apply(preprocessing_pipeline)
data.head()

,id,sentiment,sentence,preprocessed_text
0,1,1,The weather today is absolutely beautiful and ...,weather today absolutely beautiful bright
1,2,0,I can't believe how terrible the service was a...,cant believe terrible service restaurant
2,3,1,She passed her exam with flying colors and fel...,passed exam flying color felt incredibly proud
3,4,0,The project failed miserably due to poor plann...,project failed miserably due poor planning com...
4,5,1,Every morning I wake up feeling grateful and f...,every morning wake feeling grateful full energy


In [21]:
tfidf = TfidfVectorizer(max_features=5000)
text_matrix = tfidf.fit_transform(data['preprocessed_text'])
print(text_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 115 stored elements and shape (20, 112)>
  Coords	Values
  (0, 103)	0.4472135954999579
  (0, 93)	0.4472135954999579
  (0, 0)	0.4472135954999579
  (0, 5)	0.4472135954999579
  (0, 9)	0.4472135954999579
  (1, 12)	0.4472135954999579
  (1, 7)	0.4472135954999579
  (1, 91)	0.4472135954999579
  (1, 82)	0.4472135954999579
  (1, 78)	0.4472135954999579
  (2, 68)	0.38425551511623257
  (2, 34)	0.38425551511623257
  (2, 44)	0.38425551511623257
  (2, 18)	0.38425551511623257
  (2, 41)	0.3377664794152205
  (2, 52)	0.38425551511623257
  (2, 74)	0.38425551511623257
  (3, 73)	0.3779644730092272
  (3, 37)	0.3779644730092272
  (3, 62)	0.3779644730092272
  (3, 30)	0.3779644730092272
  (3, 71)	0.3779644730092272
  (3, 69)	0.3779644730092272
  (3, 19)	0.3779644730092272
  (4, 32)	0.3779644730092272
  :	:
  (15, 20)	0.4577405549330346
  (15, 66)	0.4577405549330346
  (15, 35)	0.4577405549330346
  (15, 109)	0.4577405549330346
  (16, 48)	0.4472135954999

In [22]:
import numpy as np

x = text_matrix.toarray()  
print(f"x shape: {x.shape}")
y = np.array(data['sentiment'])
print(f"y shape: {y.shape}")
print(f"Samples match: {len(x) == len(y)}")

x shape: (20, 112)
y shape: (20,)
Samples match: True


In [23]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [24]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [25]:
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score
cm = confusion_matrix(y_test, y_pred)
recall = recall_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
print("Confusion Matrix:\n", cm)
print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1)


Confusion Matrix:
 [[0 3]
 [0 1]]
Recall: 0.25
Precision: 0.0625
F1 Score: 0.1


C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
